# Entity Resolution — Frozen Benchmark V1 Evaluation
**Scope:** 144 sampled pairs from JBIDE, SAK, MNG, MC, HBASE and SERVER. Primary metrics use only fixed `SAME`/`DIFFERENT` human labels. `UNSURE` is excluded and counted.

**Important:** Pair candidates were stratified partly using resolver output. Scores describe **this selected pair benchmark**, not all possible pairs or all 30 projects. AI suggestions are not substituted for human labels. No model training or threshold tuning happens here.

In [7]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/01_Knowledge_Construction/Entity_Resolution')
GROUND_TRUTH = BASE / 'Ground_Truth' / 'ER_ground_truth_v1.xlsx'
# If you retained the original filename, this alternative is used automatically.
if not GROUND_TRUTH.exists():
    GROUND_TRUTH = BASE / 'Ground_Truth' / 'ER_FINAL_professor_review.xlsx'
RESULTS_DIR = BASE / 'Results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# Place the sealed prediction CSV in this folder (not in Ground_Truth).
PREDICTIONS = BASE / 'Resolver_Predictions' / 'ER_resolver_predictions_sealed.csv'
print('Ground truth:', GROUND_TRUTH, 'exists:', GROUND_TRUTH.exists())
print('Predictions:', PREDICTIONS, 'exists:', PREDICTIONS.exists())
if not GROUND_TRUTH.exists(): raise FileNotFoundError('Upload frozen workbook to Ground_Truth')
if not PREDICTIONS.exists(): raise FileNotFoundError('Upload ER_resolver_predictions_sealed.csv to Entity_Resolution/Resolver_Predictions')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ground truth: /content/drive/MyDrive/01_Knowledge_Construction/Entity_Resolution/Ground_Truth/ER_ground_truth_v1.xlsx exists: True
Predictions: /content/drive/MyDrive/01_Knowledge_Construction/Entity_Resolution/Resolver_Predictions/ER_resolver_predictions_sealed.csv exists: True


## 1. Load and validate human annotations
The input is restricted to the `Human annotation` sheet, not the `Evidence and proposals` sheet. Formula labels are rejected instead of silently counted as human confirmation.

In [8]:
import pandas as pd, numpy as np, openpyxl
from IPython.display import display
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from collections import Counter

wb = openpyxl.load_workbook(GROUND_TRUTH, data_only=False, read_only=True)
if 'Human annotation' not in wb.sheetnames: raise ValueError('Human annotation sheet not found')
ws = wb['Human annotation']
rows = list(ws.values)
headers = [str(x).strip() if x is not None else '' for x in rows[0]]
human = pd.DataFrame(rows[1:], columns=headers).dropna(how='all').copy()
required = ['pair_id','project','entity_type','name_a','name_b','human_label']
missing = set(required) - set(human.columns)
if missing: raise ValueError(f'Missing columns: {missing}')
for c in ['pair_id','project','entity_type','name_a','name_b']:
    human[c] = human[c].astype(str).str.strip()
human['human_label'] = human['human_label'].fillna('').astype(str).str.strip().str.upper()
formula_mask = human['human_label'].str.startswith('=')
invalid = ~human['human_label'].isin(['SAME','DIFFERENT','UNSURE'])
assert not formula_mask.any(), f'{formula_mask.sum()} formula labels detected: must be fixed human decisions'
assert not invalid.any(), f'Invalid/missing human labels: {human.loc[invalid,["pair_id","human_label"]].to_dict("records")[:10]}'
assert human['pair_id'].is_unique, 'Duplicate pair_id in human workbook'
PROJECTS = {'JBIDE','SAK','MNG','MC','HBASE','SERVER'}
assert set(human['project']) == PROJECTS, f'Unexpected project coverage: {set(human.project)}'
print('Pairs:',len(human),'Projects:', sorted(human.project.unique()))
print('Human labels:');display(human.human_label.value_counts().rename_axis('label').reset_index(name='count'))
print('Per-project coverage:');display(pd.crosstab(human.project,human.human_label))

Pairs: 144 Projects: ['HBASE', 'JBIDE', 'MC', 'MNG', 'SAK', 'SERVER']
Human labels:


,label,count
0,SAME,67
1,UNSURE,41
2,DIFFERENT,36


Per-project coverage:


human_label,DIFFERENT,SAME,UNSURE
project,,,
HBASE,9,11,4
JBIDE,5,12,7
MC,5,10,9
MNG,6,11,7
SAK,5,12,7
SERVER,6,11,7


## 2. Load sealed resolver predictions and join strictly by pair ID
A missing or duplicate prediction is an error, not an automatic `DIFFERENT` decision.

In [9]:
pred = pd.read_csv(PREDICTIONS, dtype=str).fillna('')
needed = {'pair_id','project','resolver_same'}
assert needed.issubset(pred.columns), f'Missing prediction columns: {needed-set(pred.columns)}'
for c in ['pair_id','project','resolver_same']: pred[c]=pred[c].astype(str).str.strip()
assert pred.pair_id.is_unique, 'Duplicate prediction pair_id'
assert pred.resolver_same.isin(['0','1']).all(), 'resolver_same must be 0 or 1'
assert set(pred.pair_id)==set(human.pair_id), 'Human and prediction pair IDs differ'
joined = human.merge(pred[['pair_id','project','resolver_same']],on='pair_id',suffixes=('','_pred'),validate='one_to_one')
assert (joined.project==joined.project_pred).all(), 'Project mismatch between ground truth and predictions'
joined['predicted_label']=joined.resolver_same.map({'1':'SAME','0':'DIFFERENT'})
joined['scored']=joined.human_label.isin(['SAME','DIFFERENT'])
scored=joined.loc[joined.scored].copy()
assert len(scored)>0, 'No definite labels to score'
scored['truth_positive']=(scored.human_label=='SAME').astype(int)
scored['pred_positive']=(scored.predicted_label=='SAME').astype(int)
scored['outcome']=np.select([
 (scored.truth_positive==1)&(scored.pred_positive==1),
 (scored.truth_positive==0)&(scored.pred_positive==1),
 (scored.truth_positive==1)&(scored.pred_positive==0)],['TP','FP','FN'],default='TN')
print('Scored:',len(scored),'Excluded UNSURE:',(~joined.scored).sum())
print('Resolver decisions on UNSURE cases (descriptive only):')
display(joined.loc[~joined.scored,'predicted_label'].value_counts().rename_axis('prediction').reset_index(name='count'))

Scored: 103 Excluded UNSURE: 41
Resolver decisions on UNSURE cases (descriptive only):


,prediction,count
0,DIFFERENT,36
1,SAME,5


## 3. Primary ER metrics
`SAME` is the positive class. A perfect result on a small selected subset is **not** proof of general deployment accuracy.

In [10]:
def metrics(df):
    tp=int((df.outcome=='TP').sum());fp=int((df.outcome=='FP').sum());fn=int((df.outcome=='FN').sum());tn=int((df.outcome=='TN').sum())
    p=tp/(tp+fp) if tp+fp else np.nan
    r=tp/(tp+fn) if tp+fn else np.nan
    f=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan
    return {'N':len(df),'TP':tp,'FP':fp,'FN':fn,'TN':tn,'Precision':p,'Recall':r,'F1':f,'Accuracy':(tp+tn)/len(df) if len(df) else np.nan}
overall=pd.DataFrame([metrics(scored)])
print('ENTITY RESOLUTION — FROZEN V1 PRIMARY RESULTS')
display(overall.round(3))
print('Human vs resolver confusion matrix:')
display(pd.crosstab(scored.human_label,scored.predicted_label,dropna=False))
print('Sample composition:')
display(pd.DataFrame({'category':['SAME','DIFFERENT','UNSURE'], 'count':[(joined.human_label==x).sum() for x in ['SAME','DIFFERENT','UNSURE']]}))

ENTITY RESOLUTION — FROZEN V1 PRIMARY RESULTS


,N,TP,FP,FN,TN,Precision,Recall,F1,Accuracy
0,103,67,0,0,36,1.0,1.0,1.0,1.0


Human vs resolver confusion matrix:


predicted_label,DIFFERENT,SAME
human_label,,
DIFFERENT,36,0
SAME,0,67


Sample composition:


,category,count
0,SAME,67
1,DIFFERENT,36
2,UNSURE,41


## 4. Breakdown and error audit
Undefined subgroup metrics appear as blank/NaN, not zero.

In [11]:
by_project=pd.DataFrame([{'project':k,**metrics(g)} for k,g in scored.groupby('project')]).sort_values('project')
by_type=pd.DataFrame([{'entity_type':k,**metrics(g)} for k,g in scored.groupby('entity_type')]).sort_values('entity_type')
print('By project:');display(by_project.round(3))
print('By entity type:');display(by_type.round(3))
print('False merges (FP):');display(scored.loc[scored.outcome=='FP',['pair_id','project','entity_type','name_a','name_b','human_label','predicted_label']].head(30))
print('Missed merges (FN):');display(scored.loc[scored.outcome=='FN',['pair_id','project','entity_type','name_a','name_b','human_label','predicted_label']].head(30))

By project:


,project,N,TP,FP,FN,TN,Precision,Recall,F1,Accuracy
0,HBASE,20,11,0,0,9,1.0,1.0,1.0,1.0
1,JBIDE,17,12,0,0,5,1.0,1.0,1.0,1.0
2,MC,15,10,0,0,5,1.0,1.0,1.0,1.0
3,MNG,17,11,0,0,6,1.0,1.0,1.0,1.0
4,SAK,17,12,0,0,5,1.0,1.0,1.0,1.0
5,SERVER,17,11,0,0,6,1.0,1.0,1.0,1.0


By entity type:


,entity_type,N,TP,FP,FN,TN,Precision,Recall,F1,Accuracy
0,COMPONENT,15,13,0,0,2,1.0,1.0,1.0,1.0
1,FEATURE,19,12,0,0,7,1.0,1.0,1.0,1.0
2,ISSUE,13,13,0,0,0,1.0,1.0,1.0,1.0
3,MILESTONE,8,0,0,0,8,NaN,NaN,NaN,1.0
4,PERSON,3,0,0,0,3,NaN,NaN,NaN,1.0
5,PROJECT,4,4,0,0,0,1.0,1.0,1.0,1.0
6,REQUIREMENT,22,8,0,0,14,1.0,1.0,1.0,1.0
7,SYSTEM,12,12,0,0,0,1.0,1.0,1.0,1.0
8,TEAM,7,5,0,0,2,1.0,1.0,1.0,1.0


False merges (FP):


,pair_id,project,entity_type,name_a,name_b,human_label,predicted_label


Missed merges (FN):


,pair_id,project,entity_type,name_a,name_b,human_label,predicted_label


## 5. Bootstrap uncertainty and saved outputs
Bootstrap resamples **pairs** (2,000 replicates). Because candidate pairs can share entities/documents and sampling is stratified, this is only an approximate conditional uncertainty summary—not a population confidence interval.

In [12]:
rng=np.random.default_rng(42)
arr=scored[['truth_positive','pred_positive']].to_numpy(dtype=int)
boot=[]
for _ in range(2000):
    sub=arr[rng.integers(0,len(arr),len(arr))]
    tp=((sub[:,0]==1)&(sub[:,1]==1)).sum()
    fp=((sub[:,0]==0)&(sub[:,1]==1)).sum()
    fn=((sub[:,0]==1)&(sub[:,1]==0)).sum()
    denom=2*tp+fp+fn
    if denom: boot.append(2*tp/denom)
ci=np.percentile(boot,[2.5,97.5]) if boot else [np.nan,np.nan]
print('Pair-bootstrap F1 interval (approx.):', [round(float(x),3) for x in ci])
overall['F1_bootstrap_low']=ci[0];overall['F1_bootstrap_high']=ci[1]
overall['excluded_unsure']=int((~joined.scored).sum())
overall['benchmark_pairs']=len(joined)
overall.to_csv(RESULTS_DIR/'ER_metrics_overall.csv',index=False)
by_project.to_csv(RESULTS_DIR/'ER_metrics_by_project.csv',index=False)
by_type.to_csv(RESULTS_DIR/'ER_metrics_by_type.csv',index=False)
scored.to_csv(RESULTS_DIR/'ER_scored_pairs_and_errors.csv',index=False)
joined.loc[~joined.scored].to_csv(RESULTS_DIR/'ER_excluded_unsure_pairs.csv',index=False)
summary=f"""Entity Resolution V1 — Selected-pair benchmark
Projects: {', '.join(sorted(PROJECTS))}
Pairs: {len(joined)}; scored: {len(scored)}; excluded UNSURE: {(~joined.scored).sum()}
Precision: {overall.iloc[0]['Precision']:.3f}; Recall: {overall.iloc[0]['Recall']:.3f}; F1: {overall.iloc[0]['F1']:.3f}
Approximate pair-bootstrap F1 interval: {ci[0]:.3f}–{ci[1]:.3f}
Limitations: resolver-informed candidate sampling; unresolved pairs excluded; no independent inter-annotator agreement; results do not establish population-wide accuracy or autonomous deployment readiness.
"""
(RESULTS_DIR/'ER_methodology_summary.txt').write_text(summary,encoding='utf-8')
print(summary)
print('Saved to:',RESULTS_DIR)

Pair-bootstrap F1 interval (approx.): [1.0, 1.0]
Entity Resolution V1 — Selected-pair benchmark
Projects: HBASE, JBIDE, MC, MNG, SAK, SERVER
Pairs: 144; scored: 103; excluded UNSURE: 41
Precision: 1.000; Recall: 1.000; F1: 1.000
Approximate pair-bootstrap F1 interval: 1.000–1.000
Limitations: resolver-informed candidate sampling; unresolved pairs excluded; no independent inter-annotator agreement; results do not establish population-wide accuracy or autonomous deployment readiness.

Saved to: /content/drive/MyDrive/01_Knowledge_Construction/Entity_Resolution/Results


On a stratified six-project Entity Resolution benchmark containing 144 candidate pairs, 103 pairs received definite human labels and were included in scoring. The resolver achieved pairwise Precision, Recall and F1 of 1.000, with no observed errors on the scored subset. However, 41 uncertain pairs were excluded, and candidate selection was partly informed by resolver outputs. Consequently, these results demonstrate perfect agreement on the evaluated subset but do not establish generalization to the full dataset.

The benchmark excludes uncertain cases, which may be harder to resolve. Therefore, 100% F1 does not establish that the resolver is 100% accurate on unseen data.